# Step 12: Regression with Feature Engineering
## Engineer regression-specific features and train multi-horizon models

This notebook implements feature engineering specifically for Vt regression:

**Engineered Features:**
1. **Lot-level deviation features**: `sensor_value - lot_mean_sensor` (removes lot baseline)
2. **Lot-level normalized features**: `(sensor_value - lot_mean) / lot_std` (z-scores per sensor)
3. **Position-based rolling statistics**: mean/std/min/max of previous N wafers in lot
4. **Equipment-specific baselines**: `sensor_value - equip_mean_sensor`
5. **Lot aggregate features**: lot-level mean/std/range for each sensor
6. **Interaction features**: key sensor pairs (multiplicative interactions)

**Target**: Raw Vt value (continuous prediction)

## 1. Setup and Configuration

In [28]:
import time
import json
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

start_time = time.time()
print("=" * 60)
print("Step 12: Regression with Feature Engineering")
print("=" * 60)

Step 12: Regression with Feature Engineering


In [29]:
# AUTO-DETECT PROJECT ROOT
import os
from pathlib import Path

def find_project_root(marker_file='CLAUDE.md', max_depth=5):
    """Find project root by looking for marker file"""
    current = Path.cwd()
    
    # Try current directory and parents
    for _ in range(max_depth):
        if (current / marker_file).exists():
            return current
        current = current.parent
    
    # If not found, return current working directory
    print(f"WARNING: Could not find {marker_file}, using current directory")
    return Path.cwd()

# Find and change to project root
project_root = find_project_root()
os.chdir(project_root)

print(f"Project root: {project_root}")
print(f"Current directory: {Path.cwd()}")

Project root: d:\capstone_pipeline
Current directory: d:\capstone_pipeline


In [30]:
# CONFIGURATION
vt_type = 'NFET1_VT'  # Vt type to predict
force = False  # Force rebuild
rolling_window = 5  # Number of previous wafers for rolling stats

print(f"\n[CONFIG] Vt type: {vt_type}")
print(f"[CONFIG] Rolling window: {rolling_window} wafers")
print(f"[CONFIG] Force rebuild: {force}")


[CONFIG] Vt type: NFET1_VT
[CONFIG] Rolling window: 5 wafers
[CONFIG] Force rebuild: False


In [31]:
# Setup paths
data_dir = Path("data")
output_dir = Path("outputs")
features_dir = output_dir / "features"
models_dir = output_dir / "models"
plots_dir = output_dir / "plots"
models_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

sentinel = models_dir / "12_regression_feature_eng.done"
if sentinel.exists() and not force:
    print(f"\n[OK] Feature-engineered models already trained (found {sentinel})")
    print(f"  Set force=True to rebuild")
else:
    print(f"\n[INFO] Starting feature engineering and model training...")


[INFO] Starting feature engineering and model training...


## 2. Load Base Data

In [32]:
# Load existing feature matrices
print("\nLoading base feature matrices...")
train_df = pd.read_parquet(features_dir / "train.parquet")
val_df = pd.read_parquet(features_dir / "val.parquet")
print(f"  Train size: {len(train_df)}")
print(f"  Val size: {len(val_df)}")
print(f"  Base features: {len([c for c in train_df.columns if c not in ['WAFER_SCRIBE', 'LOT_ID', 'is_outlier', 'PARAM_END_DATETIME', 'first_start_time']])}")


Loading base feature matrices...
  Train size: 13510
  Val size: 3307
  Base features: 10302


In [33]:
# Load regression targets
print("\nLoading regression targets...")
response_df = pd.read_csv(data_dir / "response_updated.csv")

# Filter and deduplicate
vt_filtered = response_df[response_df['NAME'] == vt_type][['WAFER_SCRIBE', 'LOT_ID', 'VALUE']].copy()
duplicates = len(vt_filtered) - vt_filtered['WAFER_SCRIBE'].nunique()
if duplicates > 0:
    print(f"  Found {duplicates} duplicate measurements - averaging per wafer")
    vt_filtered = vt_filtered.groupby(['WAFER_SCRIBE', 'LOT_ID'], as_index=False)['VALUE'].mean()

vt_df = vt_filtered.rename(columns={'VALUE': 'vt_value'})
print(f"  Unique wafers with {vt_type}: {len(vt_df)}")
print(f"  Vt value range: [{vt_df['vt_value'].min():.4f}, {vt_df['vt_value'].max():.4f}]")


Loading regression targets...
  Found 385 duplicate measurements - averaging per wafer
  Unique wafers with NFET1_VT: 16821
  Vt value range: [-0.7109, 1.5781]


In [34]:
# Merge targets with feature matrices
print("\nMerging targets...")
train_df = train_df.merge(vt_df[['WAFER_SCRIBE', 'vt_value']], on='WAFER_SCRIBE', how='left')
val_df = val_df.merge(vt_df[['WAFER_SCRIBE', 'vt_value']], on='WAFER_SCRIBE', how='left')

# Drop rows with missing targets
train_df = train_df.dropna(subset=['vt_value'])
val_df = val_df.dropna(subset=['vt_value'])

# Drop old classification target
if 'is_outlier' in train_df.columns:
    train_df = train_df.drop(columns=['is_outlier'])
    val_df = val_df.drop(columns=['is_outlier'])

print(f"  Train size after merge: {len(train_df)}")
print(f"  Val size after merge: {len(val_df)}")


Merging targets...
  Train size after merge: 13514
  Val size after merge: 3307


In [35]:
# Identify sensor feature columns (exclude metadata)
metadata_cols = ['WAFER_SCRIBE', 'LOT_ID', 'vt_value', 'PARAM_END_DATETIME']
base_feature_cols = [c for c in train_df.columns if c not in metadata_cols]

# Separate sensor features from other features
sensor_cols = [c for c in base_feature_cols if ('__MEAN' in c or '__STD' in c) and not c.startswith('LOT_ID')]
print(f"\n  Identified {len(sensor_cols)} sensor/SPC measurement columns for feature engineering")


  Identified 9888 sensor/SPC measurement columns for feature engineering


## 3. Feature Engineering - Lot-Level Deviation Features

In [36]:
# Combine train and val temporarily for consistent lot statistics
print("\n" + "=" * 60)
print("Feature Engineering Step 1: Lot-Level Deviations")
print("=" * 60)

combined_df = pd.concat([train_df, val_df], ignore_index=True)
print(f"Combined dataset size: {len(combined_df)}")

# Compute lot-level statistics for each sensor
print(f"\nComputing lot-level statistics for {len(sensor_cols)} sensors...")
print("  (This may take a few minutes)")

# Select top sensors to avoid memory issues (use top 50 by variance)
sensor_variances = train_df[sensor_cols].var().sort_values(ascending=False)
top_sensors = sensor_variances.head(50).index.tolist()
print(f"  Using top {len(top_sensors)} sensors by variance for deviation features")

lot_deviation_features = []

for sensor in tqdm(top_sensors, desc="Lot deviations"):
    # Compute lot-level mean and std
    lot_stats = combined_df.groupby('LOT_ID')[sensor].agg(['mean', 'std']).reset_index()
    lot_stats.columns = ['LOT_ID', f'{sensor}__lot_mean', f'{sensor}__lot_std']
    
    # Merge back
    combined_df = combined_df.merge(lot_stats, on='LOT_ID', how='left')
    
    # Create deviation feature (value - lot_mean)
    combined_df[f'{sensor}__lot_dev'] = combined_df[sensor] - combined_df[f'{sensor}__lot_mean']
    
    # Create normalized deviation (z-score within lot)
    lot_std = combined_df[f'{sensor}__lot_std'].replace(0, 1.0)  # Avoid division by zero
    combined_df[f'{sensor}__lot_zscore'] = combined_df[f'{sensor}__lot_dev'] / lot_std
    
    lot_deviation_features.extend([
        f'{sensor}__lot_mean',
        f'{sensor}__lot_std',
        f'{sensor}__lot_dev',
        f'{sensor}__lot_zscore'
    ])

print(f"\n[OK] Created {len(lot_deviation_features)} lot-level deviation features")


Feature Engineering Step 1: Lot-Level Deviations
Combined dataset size: 16821

Computing lot-level statistics for 9888 sensors...
  (This may take a few minutes)
  Using top 50 sensors by variance for deviation features


Lot deviations: 100%|██████████| 50/50 [00:00<00:00, 99.25it/s] 


[OK] Created 200 lot-level deviation features


## 4. Feature Engineering - Rolling Statistics Within Lot

In [37]:
print("\n" + "=" * 60)
print("Feature Engineering Step 2: Rolling Statistics (Position-Based)")
print("=" * 60)

# Sort by lot and process time to get position within lot
combined_df = combined_df.sort_values(['LOT_ID', 'PARAM_END_DATETIME']).reset_index(drop=True)

rolling_features = []

# Use top 20 sensors for rolling stats (to avoid memory issues)
top_sensors_rolling = sensor_variances.head(20).index.tolist()
print(f"Computing rolling statistics (window={rolling_window}) for {len(top_sensors_rolling)} sensors...")

for sensor in tqdm(top_sensors_rolling, desc="Rolling stats"):
    # Rolling mean of previous N wafers within same lot
    combined_df[f'{sensor}__roll_mean'] = combined_df.groupby('LOT_ID')[sensor].transform(
        lambda x: x.rolling(window=rolling_window, min_periods=1).mean().shift(1)
    )
    
    # Rolling std of previous N wafers within same lot
    combined_df[f'{sensor}__roll_std'] = combined_df.groupby('LOT_ID')[sensor].transform(
        lambda x: x.rolling(window=rolling_window, min_periods=1).std().shift(1)
    )
    
    # Deviation from rolling mean
    combined_df[f'{sensor}__roll_dev'] = combined_df[sensor] - combined_df[f'{sensor}__roll_mean']
    
    rolling_features.extend([
        f'{sensor}__roll_mean',
        f'{sensor}__roll_std',
        f'{sensor}__roll_dev'
    ])

# Fill NaN in rolling features with 0 (for first wafers in lot)
combined_df[rolling_features] = combined_df[rolling_features].fillna(0)

print(f"\n[OK] Created {len(rolling_features)} rolling statistic features")


Feature Engineering Step 2: Rolling Statistics (Position-Based)
Computing rolling statistics (window=5) for 20 sensors...


Rolling stats: 100%|██████████| 20/20 [00:05<00:00,  3.48it/s]


[OK] Created 60 rolling statistic features


# Prepare metadata columns
metadata_cols = ['WAFER_SCRIBE', 'LOT_ID', 'vt_value', 'PARAM_END_DATETIME']
feature_cols = [col for col in train_df.columns if col not in metadata_cols]

print(f"\n  Total available features: {len(feature_cols)}")

In [38]:
print("\n" + "=" * 60)
print("Feature Engineering Step 3: Equipment-Specific Baselines")
print("=" * 60)

# Find equipment columns
equip_cols = [c for c in base_feature_cols if '__EQUIP' in c]
print(f"Found {len(equip_cols)} equipment columns")

equipment_features = []

# For each equipment column, compute equipment-specific baselines for corresponding sensors
# (This is computationally expensive, so limit to top 10 sensors)
top_sensors_equip = sensor_variances.head(10).index.tolist()

if len(equip_cols) > 0:
    print(f"Computing equipment baselines for {len(top_sensors_equip)} sensors...")
    
    for sensor in tqdm(top_sensors_equip, desc="Equipment baselines"):
        # Extract step name from sensor column
        # Format: SensorName__StepName__MEAN or __STD
        parts = sensor.split('__')
        if len(parts) >= 2:
            step_name = parts[1]
            equip_col = f"{step_name}__EQUIP"
            
            if equip_col in combined_df.columns:
                # Compute equipment-level mean
                equip_stats = combined_df.groupby(equip_col)[sensor].mean().reset_index()
                equip_stats.columns = [equip_col, f'{sensor}__equip_mean']
                
                # Merge back
                combined_df = combined_df.merge(equip_stats, on=equip_col, how='left')
                
                # Create deviation from equipment baseline
                combined_df[f'{sensor}__equip_dev'] = combined_df[sensor] - combined_df[f'{sensor}__equip_mean']
                
                equipment_features.extend([
                    f'{sensor}__equip_mean',
                    f'{sensor}__equip_dev'
                ])
    
    print(f"\n[OK] Created {len(equipment_features)} equipment-specific features")
else:
    print("\n[INFO] No equipment columns found, skipping equipment baselines")


Feature Engineering Step 3: Equipment-Specific Baselines
Found 0 equipment columns

[INFO] No equipment columns found, skipping equipment baselines


## 6. Feature Engineering - Interaction Features

In [39]:
print("\n" + "=" * 60)
print("Feature Engineering Step 4: Sensor Interaction Features")
print("=" * 60)

# Create interaction features between top sensors
top_sensors_interact = sensor_variances.head(10).index.tolist()
interaction_features = []

print(f"Creating interaction features for top {len(top_sensors_interact)} sensors...")

# Create pairwise interactions (multiplicative)
for i, sensor1 in enumerate(tqdm(top_sensors_interact, desc="Interactions")):
    for j, sensor2 in enumerate(top_sensors_interact):
        if j > i:  # Only upper triangle to avoid duplicates
            interaction_name = f'{sensor1}__X__{sensor2}'
            combined_df[interaction_name] = combined_df[sensor1] * combined_df[sensor2]
            interaction_features.append(interaction_name)

print(f"\n[OK] Created {len(interaction_features)} interaction features")


Feature Engineering Step 4: Sensor Interaction Features
Creating interaction features for top 10 sensors...


Interactions: 100%|██████████| 10/10 [00:00<00:00, 117.84it/s]


[OK] Created 45 interaction features


## 7. Prepare Enhanced Feature Matrices

In [40]:
print("\n" + "=" * 60)
print("Preparing Enhanced Feature Matrices")
print("=" * 60)

# Collect all engineered features
engineered_features = (
    lot_deviation_features + 
    rolling_features + 
    equipment_features + 
    interaction_features
)

print(f"\nTotal base features: {len(base_feature_cols)}")
print(f"Total engineered features: {len(engineered_features)}")
print(f"Total features: {len(base_feature_cols) + len(engineered_features)}")

# Split back into train/val
train_enhanced = combined_df[combined_df['WAFER_SCRIBE'].isin(train_df['WAFER_SCRIBE'])].copy()
val_enhanced = combined_df[combined_df['WAFER_SCRIBE'].isin(val_df['WAFER_SCRIBE'])].copy()

print(f"\nTrain size: {len(train_enhanced)}")
print(f"Val size: {len(val_enhanced)}")

# Define all feature columns (base + engineered)
all_feature_cols = base_feature_cols + engineered_features


Preparing Enhanced Feature Matrices

Total base features: 10302
Total engineered features: 305
Total features: 10607

Train size: 13514
Val size: 3307


## 8. Load Column-Step Mapping and Define Horizons

In [41]:
# Load column-step mapping
print("\nLoading column-step mapping...")
with open(features_dir / "column_step_map.json", 'r') as f:
    column_step_map = json.load(f)

# Assign SeqNo=0 to all engineered features (available from start)
# In practice, engineered features inherit the SeqNo of their base sensor
# For simplicity, we'll map each engineered feature to its parent sensor's SeqNo

extended_step_map = column_step_map.copy()

for feat in engineered_features:
    # Extract parent sensor name
    # Format: SensorName__StepName__MEAN__lot_dev
    # Find the base sensor column
    base_sensor = None
    for sensor in sensor_cols:
        if feat.startswith(sensor):
            base_sensor = sensor
            break
    
    if base_sensor and base_sensor in extended_step_map:
        extended_step_map[feat] = extended_step_map[base_sensor]
    else:
        # Default to SeqNo=0 if no parent found
        extended_step_map[feat] = 0

print(f"Extended step map size: {len(extended_step_map)}")


Loading column-step mapping...
Extended step map size: 10607


In [42]:
# Define horizons
step_seq_df = pd.read_csv(data_dir / "step_seq.csv")
all_seqnos = sorted(step_seq_df['SeqNo'].unique())

percentiles = np.percentile(all_seqnos, [10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
horizons = sorted(set([int(p) for p in percentiles]))

print(f"\nTraining at {len(horizons)} horizons: {horizons}")


Training at 10 horizons: [24, 48, 72, 96, 120, 144, 168, 192, 216, 240]


In [43]:
# Extract targets
y_train = train_enhanced['vt_value']
y_val = val_enhanced['vt_value']

print(f"\nTarget statistics (train):")
print(f"  Mean: {y_train.mean():.4f}")
print(f"  Std: {y_train.std():.4f}")
print(f"  Range: [{y_train.min():.4f}, {y_train.max():.4f}]")


Target statistics (train):
  Mean: 0.5051
  Std: 0.1585
  Range: [-0.7109, 1.5781]


## 9. Multi-Horizon Training with Enhanced Features

In [44]:
# Train models
horizon_results = []

print("\n" + "=" * 80)
print("Training Multi-Horizon Regression Models (Enhanced Features)")
print("=" * 80)
print(f"{'Horizon':<10} {'Features':<10} {'RMSE':<12} {'MAE':<12} {'R²':<10}")
print("-" * 80)

for k in tqdm(horizons, desc="Horizons"):
    # Filter features available at horizon k
    available_features = [
        col for col in all_feature_cols
        if col in extended_step_map and extended_step_map[col] <= k
    ]
    
    if len(available_features) == 0:
        print(f"\n  WARNING: No features at horizon {k}, skipping...")
        continue
    
    # Prepare feature matrices
    X_train = train_enhanced[available_features].copy()
    X_val = val_enhanced[available_features].copy()
    
    # Handle infinite/NaN values in engineered features
    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_val = X_val.replace([np.inf, -np.inf], np.nan).fillna(0)
    
    # Identify categorical features more robustly
    cat_features = []
    for col in available_features:
        # Check by name pattern
        is_categorical_by_name = (
            col.startswith('LOT_ID') or
            '__EQUIP' in col or
            '__POSITION' in col
        )
        
        # Check by data type
        is_categorical_by_type = (
            X_train[col].dtype == 'object' or
            X_train[col].dtype.name == 'string' or
            str(X_train[col].dtype).startswith('str')
        )
        
        # Check by content (if numeric, check if it's actually discrete values)
        is_categorical_by_content = False
        if not is_categorical_by_type and X_train[col].dtype in ['int64', 'float64']:
            # If column has very few unique values and they look like IDs, treat as categorical
            n_unique = X_train[col].nunique()
            if n_unique < 100 and n_unique < len(X_train) * 0.05:
                # Check if values look like IDs (integers or have specific patterns)
                sample_val = X_train[col].dropna().iloc[0] if len(X_train[col].dropna()) > 0 else None
                if sample_val is not None:
                    is_categorical_by_content = True
        
        if is_categorical_by_name or is_categorical_by_type or is_categorical_by_content:
            if col in X_train.columns:
                cat_features.append(col)
                # Convert to string for CatBoost
                X_train[col] = X_train[col].fillna('UNKNOWN').astype(str)
                X_val[col] = X_val[col].fillna('UNKNOWN').astype(str)
    
    # Create pools
    train_pool = Pool(X_train, y_train, cat_features=cat_features)
    val_pool = Pool(X_val, y_val, cat_features=cat_features)
    
    # Train model
    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function='RMSE',
        eval_metric='RMSE',
        early_stopping_rounds=30,
        random_seed=42,
        verbose=0
    )
    
    model.fit(train_pool, eval_set=val_pool)
    
    # Evaluate
    y_pred = model.predict(X_val)
    
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    
    # Save model
    model_file = models_dir / f"regression_feature_eng_horizon_{k:03d}_model.cbm"
    model.save_model(str(model_file))
    
    # Store results
    horizon_results.append({
        'horizon': int(k),
        'n_features': len(available_features),
        'n_base_features': len([f for f in available_features if f in base_feature_cols]),
        'n_engineered_features': len([f for f in available_features if f in engineered_features]),
        'rmse': float(rmse),
        'mae': float(mae),
        'r2': float(r2)
    })
    
    print(f"k={k:<7} {len(available_features):<10} {rmse:<12.6f} {mae:<12.6f} {r2:<10.4f}")

print("-" * 80)


Training Multi-Horizon Regression Models (Enhanced Features)
Horizon    Features   RMSE         MAE          R²        
--------------------------------------------------------------------------------


Horizons:  10%|█         | 1/10 [00:11<01:47, 11.93s/it]

k=24      1774       0.157368     0.116086     -0.0260   


Horizons:  20%|██        | 2/10 [01:19<05:57, 44.73s/it]

k=48      4558       0.155039     0.114350     0.0041    


...

k=192     9750       0.144863     0.103849     0.1305    


Horizons:  90%|█████████ | 9/10 [32:05<05:15, 315.07s/it]

k=216     10152      0.139892     0.099904     0.1892    


Horizons: 100%|██████████| 10/10 [35:55<00:00, 215.52s/it]

k=240     10607      0.142772     0.101958     0.1555    
--------------------------------------------------------------------------------


## 10. Save Results and Generate Plots

In [45]:
# Save results
results_file = output_dir / "regression_feature_eng_horizon_results.json"
with open(results_file, 'w') as f:
    json.dump({
        'vt_type': vt_type,
        'target': 'raw_vt_value',
        'feature_engineering': {
            'lot_deviation': True,
            'rolling_stats': True,
            'equipment_baselines': True,
            'interactions': True,
            'rolling_window': rolling_window
        },
        'horizons': horizon_results
    }, f, indent=2)

print(f"\n[OK] Saved {results_file.name}")


[OK] Saved regression_feature_eng_horizon_results.json


In [46]:
# Summary
print("\n" + "=" * 60)
print("Feature-Engineered Regression Summary")
print("=" * 60)
print(f"Vt type: {vt_type}")
print(f"Total horizons: {len(horizon_results)}")
print(f"\nFeature Breakdown (final horizon):")
final_result = horizon_results[-1]
print(f"  Base features: {final_result['n_base_features']}")
print(f"  Engineered features: {final_result['n_engineered_features']}")
print(f"  Total features: {final_result['n_features']}")
print(f"\nPerformance range:")
print(f"  RMSE: {min(r['rmse'] for r in horizon_results):.6f} to {max(r['rmse'] for r in horizon_results):.6f}")
print(f"  R²: {min(r['r2'] for r in horizon_results):.4f} to {max(r['r2'] for r in horizon_results):.4f}")

best = max(horizon_results, key=lambda r: r['r2'])
print(f"\nBest R²: {best['r2']:.4f} at horizon {best['horizon']}")


Feature-Engineered Regression Summary
Vt type: NFET1_VT
Total horizons: 10

Feature Breakdown (final horizon):
  Base features: 10302
  Engineered features: 305
  Total features: 10607

Performance range:
  RMSE: 0.138906 to 0.157368
  R²: -0.0260 to 0.2006

Best R²: 0.2006 at horizon 168


In [47]:
# Load final model for evaluation
final_horizon = max(r['horizon'] for r in horizon_results)
final_model = CatBoostRegressor()
final_model.load_model(str(models_dir / f"regression_feature_eng_horizon_{final_horizon:03d}_model.cbm"))

# Get final features
final_features = [
    col for col in all_feature_cols
    if col in extended_step_map and extended_step_map[col] <= final_horizon
]

X_val_final = val_enhanced[final_features].copy()
X_val_final = X_val_final.replace([np.inf, -np.inf], np.nan).fillna(0)

# Handle categoricals with same robust logic
for col in final_features:
    # Check by name pattern
    is_categorical_by_name = (
        col.startswith('LOT_ID') or
        '__EQUIP' in col or
        '__POSITION' in col
    )
    
    # Check by data type
    is_categorical_by_type = (
        X_val_final[col].dtype == 'object' or
        X_val_final[col].dtype.name == 'string' or
        str(X_val_final[col].dtype).startswith('str')
    )
    
    if is_categorical_by_name or is_categorical_by_type:
        X_val_final[col] = X_val_final[col].fillna('UNKNOWN').astype(str)

y_pred_final = final_model.predict(X_val_final)

print(f"\n[INFO] Generated predictions for final horizon {final_horizon}")

CatBoostError: Invalid type for cat_feature[non-default value idx=0,feature_idx=29]=0.9999233775189627 : cat_features must be integer or string, real number values and NaN values should be converted to string.

In [ ]:
# Plot: Predicted vs Actual
print("\n[INFO] Generating plots...")
plt.figure(figsize=(10, 8))
plt.scatter(y_val, y_pred_final, alpha=0.5, s=20)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Vt Value', fontsize=12)
plt.ylabel('Predicted Vt Value', fontsize=12)
plt.title(f'Predicted vs Actual Vt (Feature Engineered) - Horizon {final_horizon}\nR² = {r2_score(y_val, y_pred_final):.4f}', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(plots_dir / "regression_feature_eng_pred_vs_actual.png", dpi=300, bbox_inches='tight')
plt.close()

print(f"[OK] Saved prediction plot")

In [ ]:
# Plot: Feature importance
importance = final_model.get_feature_importance()
importance_df = pd.DataFrame({
    'feature': final_features,
    'importance': importance
}).sort_values('importance', ascending=False).head(30)

# Color by feature type
colors = []
for feat in importance_df['feature']:
    if any(eng_feat in feat for eng_feat in ['__lot_', '__roll_', '__equip_', '__X__']):
        colors.append('orange')  # Engineered features
    elif '__SPC_' in feat:
        colors.append('green')
    else:
        colors.append('blue')

plt.figure(figsize=(12, 10))
plt.barh(range(len(importance_df)), importance_df['importance'], color=colors)
plt.yticks(range(len(importance_df)), importance_df['feature'])
plt.xlabel('Feature Importance')
plt.title(f'Top 30 Features (Orange=Engineered, Blue=Sensor, Green=SPC)\nHorizon {final_horizon}')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(plots_dir / "regression_feature_eng_importance.png", dpi=300, bbox_inches='tight')
plt.close()

print(f"[OK] Saved feature importance plot")

# Print top 20
print("\nTop 20 Most Important Features:")
print("-" * 70)
for _, row in importance_df.head(20).iterrows():
    feat_type = 'ENG' if any(x in row['feature'] for x in ['__lot_', '__roll_', '__equip_', '__X__']) else 'BASE'
    print(f"{row['feature']:<55} [{feat_type}] {row['importance']:>8.2f}")
print("-" * 70)

In [ ]:
# Save evaluation summary
summary_file = output_dir / "regression_feature_eng_evaluation_summary.txt"
with open(summary_file, 'w') as f:
    f.write("=" * 60 + "\n")
    f.write("Feature-Engineered Regression Evaluation Summary\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Vt Type: {vt_type}\n")
    f.write(f"Final Horizon: {final_horizon}\n")
    f.write(f"Total Features: {len(final_features)}\n")
    f.write(f"  Base: {final_result['n_base_features']}\n")
    f.write(f"  Engineered: {final_result['n_engineered_features']}\n\n")
    
    f.write("Performance (Final Horizon):\n")
    f.write("-" * 40 + "\n")
    f.write(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_pred_final)):.6f}\n")
    f.write(f"MAE:  {mean_absolute_error(y_val, y_pred_final):.6f}\n")
    f.write(f"R²:   {r2_score(y_val, y_pred_final):.4f}\n\n")
    
    f.write("Horizon Summary:\n")
    f.write("-" * 70 + "\n")
    f.write(f"{'Horizon':<10} {'Base':<8} {'Eng':<8} {'Total':<8} {'RMSE':<12} {'R²':<10}\n")
    f.write("-" * 70 + "\n")
    for r in horizon_results:
        f.write(f"{r['horizon']:<10} {r['n_base_features']:<8} {r['n_engineered_features']:<8} {r['n_features']:<8} {r['rmse']:<12.6f} {r['r2']:<10.4f}\n")
    f.write("-" * 70 + "\n")

print(f"\n[OK] Saved {summary_file.name}")

In [ ]:
# Mark complete
sentinel.touch()

elapsed = time.time() - start_time
print(f"\n[OK] Feature engineering and training complete in {elapsed:.2f}s")
print("=" * 60)